# Telling cats from dogs with neural networks

This is the first half of the project: an image classification task on the *cats vs. dogs* dataset. I build and compare three models, each better suited to images than the last.

1. A simple **multilayer perceptron (MLP)** as a baseline,
2. the same MLP trained with **data augmentation**, and
3. a small **convolutional neural network (CNN)**.

Finally, I vary two hyperparameters, the number of training epochs and the learning rate, to see how much they affect performance.

I first prototyped the code in Google Colab, then ran the final training runs on the university GPU cluster, submitting each run as a SLURM batch job on a single NVIDIA GPU node. This gave more consistent timings than a hosted notebook. Either way, a GPU runtime is strongly advised.

## 1. Setup

PyTorch Lightning is not pre-installed on Colab, so we add it first and then import everything we need. The network is powered by PyTorch and Lightning, the images are loaded using torchvision, and the plots are generated with matplotlib.

In [ ]:
# PyTorch Lightning is not pre-installed on Colab, so we install it first.
# This is a lightweight wrapper for PyTorch that handles the training loop.
!pip install pytorch-lightning

In [ ]:
# Matplotlib for the bar charts at the end, numpy for the little bit of array math.
from matplotlib import pyplot as plt
import numpy as np

# PyTorch is the deep-learning library, and Lightning sits on top to run the
# training, validation, and test loops so we don't write them by hand.
import torch
import torch.nn as nn
from torch.nn import functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer

# These load the images from folders and turn them into batches the network can read.
from torch.utils.data import random_split, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

# A ready-made accuracy metric, so we don't count correct predictions ourselves.
from torchmetrics.functional.classification.accuracy import accuracy

## 2. Loading the data

The images are in a zip file. To access it, either mount Google Drive or upload 'dogs-vs-cat-small.zip' directly to the session and unzip it. After unzipping, there is a 'dogs-vs-cat-small' folder with 'train', 'validation', and 'test' subfolders, each divided into 'cats' and 'dogs'.

Kaggle: https://www.kaggle.com/datasets/alinasir1596/catvsdog-small-dataset

In [ ]:
import os
# The images live in a zip file. We either mount Google Drive to reach it, or
# upload the zip straight into the Colab session and unzip it below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab wipes its disk between sessions, so the images must be put in place
# every time you start the notebook. This cell looks for the zip in the usual
# spots (uploaded into the session, or kept in Google Drive) and unzips it. If
# the data folder is already there it skips, so the cell is safe to re-run.
import os

# Where the unzipped images should end up.
data_folder = 'dogs-vs-cat-small'

# The places the zip might be sitting.
possible_zip_paths = [
    'dogs-vs-cat-small.zip',                              # current folder
    '/content/dogs-vs-cat-small.zip',                     # uploaded into Colab
    '/content/drive/MyDrive/dogs-vs-cat-small.zip',       # in your Drive
]

if os.path.isdir(data_folder + '/train'):
    print("Data is already unzipped, nothing to do.")
else:
    # Use the first path that actually exists.
    zip_path = None
    for path in possible_zip_paths:
        if os.path.exists(path):
            zip_path = path
            break

    if zip_path is None:
        print("Could not find dogs-vs-cat-small.zip in any of these places:")
        for path in possible_zip_paths:
            print("   ", path)
        print()
        print("Fix: click the folder icon on the left to open the Files panel and")
        print("upload dogs-vs-cat-small.zip, OR put it in your Drive and run the")
        print("drive.mount cell above. Then run this cell again.")
    else:
        print("Found the zip at:", zip_path, "- unzipping...")
        !unzip -q "{zip_path}"
        print("Done.")

## 3. Settings and hyperparameters

All of the notebook's knobs are combined in one place:

learning rate: how big a step the optimizer makes (kept small to avoid overshoot)
batch size: determines how many images are sent via the network at once. The image size is set to 224×224 with 3 color channels.
Mean / std: the standard ImageNet values used to normalize the pixels.
Hidden dim: neurons in the MLP's hidden layer.
Number of classes: two (cat or dog)

In [ ]:
# A quick transform used only to peek at the data below: resize every image to
# 224x224 and turn it into a tensor. The real transforms, with normalization and
# optional augmentation, are defined inside the DataModule.
manual_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
# How big a step the optimizer takes each update. Small on purpose: too large and
# training overshoots and never settles.
learning_rate = 0.001

# How many images go through the network at once before it updates its weights.
batch_size = 32

# The data already comes split into train / validation / test folders, each with
# a 'cats' and a 'dogs' sub-folder. We load the train folder here mainly to reuse
# its class names ('cats', 'dogs') for the plots later.
dataset = ImageFolder(root='dogs-vs-cat-small/train', transform=manual_transform)

# Sizes of each part, for reference (the folders already enforce these).
train_size = 2000      # 1000 cats + 1000 dogs
val_size = 1000        # 500 cats + 500 dogs
dataset_size = 3000    # train + validation

# Every image is forced to 224 x 224 pixels with 3 color channels (red, green, blue).
input_width = 224
input_height = 224
input_channels = 3

# Standard ImageNet mean/std. We subtract the mean and divide by the std so the
# pixel values sit in a small, consistent range, which makes training smoother.
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# One hidden layer of 64 neurons, and two classes to tell apart (cat or dog).
hidden_dim = 64
num_classes = 2

# Catch mistakes early instead of a confusing error later.
assert train_size + val_size == dataset_size, "Oh no! The sum of the train and validation sets must be equal the whole dataset size!"
assert input_width == input_height, "Oh no! The input width should be the same as the input height!"

## 4. The data pipeline

A Lightning *DataModule* keeps all of the data in one place: the transforms and how the images are batched. Validation and test images are only resized and normalized. Training images can optionally be augmented (Section 8). Putting this in a single class lets every model below reuse the exact same pipeline, which keeps the comparisons fair.

In [ ]:
class MyCustomDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = "dogs-vs-cat-small", augment: bool = False):
        super().__init__()
        self.data_dir = data_dir
        self.augment = augment

        # For validation and test images we never change the picture: resize, turn into
        # a tensor, and normalize. The evaluation has to stay honest.
        self.eval_transform = transforms.Compose([
            transforms.Resize((input_width, input_height)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

        # For training images we can optionally add augmentation: randomly flip
        # left-right, rotate a little, and nudge the brightness and contrast. The network
        # sees slightly different versions of each image, so it is harder for it to
        # memorize the training set.
        if self.augment:
            self.train_transform = transforms.Compose([
                transforms.Resize((input_width, input_height)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        else:
            # No augmentation: training images get the same plain transform as the rest.
            self.train_transform = self.eval_transform

    def prepare_data(self):
        # Nothing to download here, the images are already unzipped on disk.
        pass

    def setup(self, stage=None):
        # ImageFolder reads the sub-folder names ('cats', 'dogs') and uses them as the
        # labels automatically. Only the training set gets the augmented transform, while
        # validation and test stay untouched.
        if stage == "fit" or stage is None:
            self.dataset_train = ImageFolder(root=self.data_dir + "/train", transform=self.train_transform)
            self.dataset_val = ImageFolder(root=self.data_dir + "/validation", transform=self.eval_transform)

        if stage == "test" or stage is None:
            self.dataset_test = ImageFolder(root=self.data_dir + "/test", transform=self.eval_transform)

    def train_dataloader(self):
        # Shuffle the training data so the network doesn't see all cats and then all
        # dogs, which would bias each update.
        return DataLoader(self.dataset_train, batch_size=batch_size, shuffle=True)

    def val_dataloader(self):
        # No need to shuffle when we're only measuring performance.
        return DataLoader(self.dataset_val, batch_size=batch_size)

    def test_dataloader(self):
        return DataLoader(self.dataset_test, batch_size=batch_size)

## 5. The baseline MLP

The baseline is a plain MLP of fully connected layers. It flattens each image into one long vector and passes it through a single hidden layer of 64 neurons with a ReLU activation. Because flattening discards which pixels were neighbors, the network cannot recognize shapes or textures, so I expect it to be weak and treat it only as a baseline.

In [ ]:
class MyCustomModel(LightningModule):
    def __init__(self, lr=learning_rate):
        super().__init__()

        # Remember the learning rate so we can later train copies with different rates
        # and compare them.
        self.lr = lr

        # During testing we store the true labels and predictions here, so we can count
        # how many of each class were right afterwards.
        self.ground_truth = []
        self.predictions = []

        # The MLP itself: flatten the image to one long vector, one hidden layer with a
        # ReLU (which keeps the positive part and zeros the rest), and a final layer
        # giving one score per class.
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_channels * input_width * input_height, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        # Run an image (or a batch of images) through the network.
        x = self.model(x)
        return x

    def training_step(self, batch, batch_nb):
        # One training batch: predict, measure the loss, log the accuracy.
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)

        preds = torch.argmax(outputs, dim=1)   # the class with the highest score
        acc = accuracy(preds, labels, task="multiclass", num_classes=num_classes)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_nb):
        # Same as training, but on the validation set and without learning from it.
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)

        preds = torch.argmax(outputs, dim=1)
        acc = accuracy(preds, labels, task="multiclass", num_classes=num_classes)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_nb):
        # Final evaluation on the held-out test set.
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)

        preds = torch.argmax(outputs, dim=1)
        acc = accuracy(preds, labels, task="multiclass", num_classes=num_classes)

        # Keep the labels and predictions so we can break the score down per class.
        self.ground_truth.append(labels.cpu())
        self.predictions.append(preds.cpu())

        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        # Adam is a solid default optimizer that adapts the step size as it goes.
        return torch.optim.Adam(self.parameters(), lr=self.lr)

## 6. Training and testing the baseline

We train for 10 epochs and then evaluate on the held-out test set.

In [ ]:
# Build a fresh network.
model = MyCustomModel()

# Point the data module at our unzipped folder (no augmentation for the baseline).
datamodule = MyCustomDataModule(data_dir='dogs-vs-cat-small')

# One epoch = one full pass over the training images. 10 is a reasonable amount:
# enough to learn something, not so much that we wait forever.
trainer = Trainer(max_epochs=10)

# Actually train the model.
trainer.fit(model, datamodule=datamodule)

In [ ]:
# Run the trained model on the test set and print its loss and accuracy.
trainer.test(model, datamodule=datamodule)

## 7. Results per class

Overall accuracy is a single number, but it is more useful to see how many cats and dogs the model identified correctly. The bar chart below shows the correct and incorrect predictions for each class.

In [ ]:
# Quick check that we saved labels and predictions during testing. They come in
# one chunk per batch, so the length is the number of test batches.
print("Ground truth length: ", len(model.ground_truth))
print("Predictions length: ", len(model.predictions))

print("Ground truth sample: ", model.ground_truth[0][:10])
print("Predictions sample: ", model.predictions[0][:10])

In [ ]:
# Count, for each class, how many test images were classified correctly and how
# many were missed, by comparing each prediction to its true label.
num_classes = 2  # cats, dogs
correct = [0] * num_classes
wrong = [0] * num_classes

for batch_gt, batch_pred in zip(model.ground_truth, model.predictions):
    for gt, pred in zip(batch_gt, batch_pred):
        gt_i = int(gt)
        pred_i = int(pred)
        if gt_i == pred_i:
            correct[gt_i] += 1
        else:
            wrong[gt_i] += 1

print("Correct per class:", correct)
print("Wrong per class:  ", wrong)

In [ ]:
# Draw a green (correct) and red (wrong) bar for each class, to see at a glance
# whether the model struggles more with cats or with dogs.
class_names = dataset.classes  # ['cats', 'dogs'], taken from the folder names
class_labels = np.arange(len(class_names))

fig = plt.figure(figsize=(6, 4))
ax = fig.add_axes([0, 0, 1, 1], title='Correct=green, Wrong=red')

# Shift the two bars slightly left/right so they sit side by side.
ax.bar(class_labels - 0.2, correct, width=0.4, color='g', label='Correct')
ax.bar(class_labels + 0.2, wrong,  width=0.4, color='r', label='Wrong')

ax.set_xticks(class_labels)
ax.set_xticklabels(class_names)
ax.set_ylabel("Number of samples")
ax.legend()
plt.show()

## 8. Improvement: data augmentation

The brief asked for one strategy to improve the image results, and I chose **data augmentation**. Each training image is randomly flipped (from left to right), rotated, and altered for brightness and contrast. Since an image is virtually never encountered twice in exactly the same form, this prevents the network from memorizing the training set, which usually improves generalization. The validation and test images remain untouched, and all other settings are identical to the baseline, so any difference in accuracy is due only to the augmentation.

In [ ]:
# A fresh network (so it starts from scratch, not from the baseline's weights)
# and the same data module with augmentation switched on.
augmented_model = MyCustomModel()
augmented_datamodule = MyCustomDataModule(data_dir='dogs-vs-cat-small', augment=True)

# Exactly the same training settings as the baseline, so the comparison is fair.
augmented_trainer = Trainer(max_epochs=10)
augmented_trainer.fit(augmented_model, datamodule=augmented_datamodule)

In [ ]:
# Test the augmented model on the same untouched test set.
augmented_trainer.test(augmented_model, datamodule=augmented_datamodule)

In [ ]:
# Same per-class counting as before, this time for the augmented model.
correct_aug = [0] * num_classes
wrong_aug = [0] * num_classes

for batch_gt, batch_pred in zip(augmented_model.ground_truth, augmented_model.predictions):
    for true_label, predicted_label in zip(batch_gt, batch_pred):
        if int(true_label) == int(predicted_label):
            correct_aug[int(true_label)] += 1
        else:
            wrong_aug[int(true_label)] += 1

print("With augmentation - correct per class:", correct_aug)
print("With augmentation - wrong per class:  ", wrong_aug)

# Same green/red bar chart for the augmented model.
fig = plt.figure(figsize=(6, 4))
ax = fig.add_axes([0, 0, 1, 1], title='Augmented model: Correct=green, Wrong=red')

ax.bar(class_labels - 0.2, correct_aug, width=0.4, color='g', label='Correct')
ax.bar(class_labels + 0.2, wrong_aug,  width=0.4, color='r', label='Wrong')

ax.set_xticks(class_labels)
ax.set_xticklabels(class_names)
ax.set_ylabel("Number of samples")
ax.legend()
plt.show()

In [ ]:
# Turn the per-class counts into one overall accuracy per model, then put them
# side by side so the effect of augmentation is obvious.
baseline_accuracy = sum(correct) / (sum(correct) + sum(wrong))
augmented_accuracy = sum(correct_aug) / (sum(correct_aug) + sum(wrong_aug))

print(f"Baseline MLP accuracy:    {baseline_accuracy:.3f}")
print(f"MLP + data augmentation:  {augmented_accuracy:.3f}")

model_names = ['Baseline MLP', 'MLP + augmentation']
accuracies = [baseline_accuracy, augmented_accuracy]

plt.figure(figsize=(6, 4))
plt.bar(model_names, accuracies, color=['gray', 'green'])
plt.ylabel('Test accuracy')
plt.title('Effect of data augmentation')
plt.ylim(0, 1)
plt.show()

## 9. A small CNN

A CNN keeps the two-dimensional structure of the image. It slides filters over the picture to detect local patterns such as corners, edges, and textures, and uses pooling layers to shrink the image step by step, which works much better for images. We train it on the same plain, un-augmented images as the baseline MLP, so the comparison isolates the effect of the architecture itself.

In [ ]:
class MyCnnModel(LightningModule):
    def __init__(self):
        super().__init__()

        # Same idea as before: collect the true labels and predictions at test time.
        self.ground_truth = []
        self.predictions = []

        # Each Conv2d looks for patterns in small 3x3 patches, and each MaxPool2d halves
        # the width and height (keeping the strongest response). We stack three of these
        # blocks, so the image shrinks 224 -> 112 -> 56 -> 28 while the feature channels
        # grow 3 -> 16 -> 32 -> 64. Then we flatten and finish with two linear layers.
        pooled_size = input_width // 8  # halved three times: 224 / 8 = 28

        self.model = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * pooled_size * pooled_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        x = self.model(x)
        return x

    def training_step(self, batch, batch_nb):
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)

        preds = torch.argmax(outputs, dim=1)
        acc = accuracy(preds, labels, task="multiclass", num_classes=num_classes)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_nb):
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)

        preds = torch.argmax(outputs, dim=1)
        acc = accuracy(preds, labels, task="multiclass", num_classes=num_classes)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_nb):
        images, labels = batch
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)

        preds = torch.argmax(outputs, dim=1)
        acc = accuracy(preds, labels, task="multiclass", num_classes=num_classes)

        # Save labels and predictions to count per class later.
        self.ground_truth.append(labels.cpu())
        self.predictions.append(preds.cpu())

        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=learning_rate)

In [ ]:
# Build the CNN and train it on the same plain data as the baseline MLP.
cnn_model = MyCnnModel()
cnn_datamodule = MyCustomDataModule(data_dir='dogs-vs-cat-small')

cnn_trainer = Trainer(max_epochs=10)
cnn_trainer.fit(cnn_model, datamodule=cnn_datamodule)

In [ ]:
# Test the CNN.
cnn_trainer.test(cnn_model, datamodule=cnn_datamodule)

In [ ]:
# Per-class right/wrong counts for the CNN.
correct_cnn = [0] * num_classes
wrong_cnn = [0] * num_classes

for batch_gt, batch_pred in zip(cnn_model.ground_truth, cnn_model.predictions):
    for true_label, predicted_label in zip(batch_gt, batch_pred):
        if int(true_label) == int(predicted_label):
            correct_cnn[int(true_label)] += 1
        else:
            wrong_cnn[int(true_label)] += 1

print("CNN - correct per class:", correct_cnn)
print("CNN - wrong per class:  ", wrong_cnn)

# Green/red bar chart for the CNN.
fig = plt.figure(figsize=(6, 4))
ax = fig.add_axes([0, 0, 1, 1], title='CNN: Correct=green, Wrong=red')

ax.bar(class_labels - 0.2, correct_cnn, width=0.4, color='g', label='Correct')
ax.bar(class_labels + 0.2, wrong_cnn,  width=0.4, color='r', label='Wrong')

ax.set_xticks(class_labels)
ax.set_xticklabels(class_names)
ax.set_ylabel("Number of samples")
ax.legend()
plt.show()

In [ ]:
# Final head-to-head: baseline MLP vs augmented MLP vs CNN, all on the same test set.
cnn_accuracy = sum(correct_cnn) / (sum(correct_cnn) + sum(wrong_cnn))

print(f"Baseline MLP accuracy:    {baseline_accuracy:.3f}")
print(f"MLP + data augmentation:  {augmented_accuracy:.3f}")
print(f"CNN accuracy:             {cnn_accuracy:.3f}")

model_names = ['Baseline MLP', 'MLP + augmentation', 'CNN']
accuracies = [baseline_accuracy, augmented_accuracy, cnn_accuracy]

plt.figure(figsize=(7, 4))
plt.bar(model_names, accuracies, color=['gray', 'green', 'steelblue'])
plt.ylabel('Test accuracy')
plt.title('MLP vs augmented MLP vs CNN')
plt.ylim(0, 1)
plt.show()

## 10. Hyperparameter study

Finally, I vary two hyperparameters to see how much they matter: the number of training **epochs** for the CNN and the **learning rate** for the baseline MLP. Each experiment retrains the model from the beginning with a new random initialization (I did not fix a random seed), so any small differences from the values above are run-to-run noise, not a real effect.

In [ ]:
# Earlier we trained the CNN for 10 epochs. Here we try a few different values to
# see how the number of epochs changes the result. Each run trains a fresh CNN
# from scratch and records its test accuracy and loss.
epochs_to_try = [5, 10, 20]
epoch_accuracies = []
epoch_losses = []

for n_epochs in epochs_to_try:
    print(f"\n=== Training the CNN for {n_epochs} epochs ===")
    cnn = MyCnnModel()
    cnn_dm = MyCustomDataModule(data_dir='dogs-vs-cat-small')
    epoch_trainer = Trainer(max_epochs=n_epochs)
    epoch_trainer.fit(cnn, datamodule=cnn_dm)

    result = epoch_trainer.test(cnn, datamodule=cnn_dm)
    epoch_accuracies.append(result[0]['test_acc'])
    epoch_losses.append(result[0]['test_loss'])

print()
for n_epochs, acc, loss in zip(epochs_to_try, epoch_accuracies, epoch_losses):
    print(f"{n_epochs:>2} epochs -> test accuracy {acc:.3f}, test loss {loss:.3f}")

# Plot accuracy and loss against the number of epochs. We expect the accuracy to
# peak early and the loss to climb as the CNN starts to overfit.
fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(11, 4))

ax_acc.plot(epochs_to_try, epoch_accuracies, marker='o')
ax_acc.set_xlabel('Epochs')
ax_acc.set_ylabel('Test accuracy')
ax_acc.set_title('CNN accuracy vs epochs')
ax_acc.set_ylim(0, 1)
ax_acc.grid(True)

ax_loss.plot(epochs_to_try, epoch_losses, marker='o', color='firebrick')
ax_loss.set_xlabel('Epochs')
ax_loss.set_ylabel('Test loss')
ax_loss.set_title('CNN loss vs epochs')
ax_loss.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Try a small, a medium and a large learning rate on the baseline MLP.
learning_rates_to_try = [0.0001, 0.001, 0.01]
lr_accuracies = []

for lr in learning_rates_to_try:
    print(f"\n=== Training the MLP with learning rate = {lr} ===")
    # Fresh model with this learning rate, fresh (un-augmented) data, same 10 epochs.
    mlp = MyCustomModel(lr=lr)
    lr_datamodule = MyCustomDataModule(data_dir='dogs-vs-cat-small')
    lr_trainer = Trainer(max_epochs=10)
    lr_trainer.fit(mlp, datamodule=lr_datamodule)

    # trainer.test returns a list with one dictionary of the logged metrics.
    result = lr_trainer.test(mlp, datamodule=lr_datamodule)
    lr_accuracies.append(result[0]['test_acc'])

# Print the learning rate next to the accuracy it reached.
print()
for lr, acc in zip(learning_rates_to_try, lr_accuracies):
    print(f"learning rate {lr:<8} -> test accuracy {acc:.3f}")

# Plot it. We use the learning rates as text labels so they are evenly spaced.
plt.figure(figsize=(7, 4))
plt.plot([str(lr) for lr in learning_rates_to_try], lr_accuracies, marker='o')
plt.xlabel('Learning rate')
plt.ylabel('Test accuracy')
plt.title('Baseline MLP: effect of the learning rate')
plt.ylim(0, 1)
plt.grid(True)
plt.show()